# 01 Inference Baseline

This notebook runs inference using trained YOLOv8 weights on validation and new images.


# Baseline Inference Notebook — YOLOv8 PPE (Hard-Hat) Detection

This notebook demonstrates inference only (no training). It:
1) Downloads the Roboflow dataset (YOLOv8 format)
2) Loads trained weights (`best.pt`)
3) Runs inference on validation images and new images
4) Saves prediction outputs into `results/evidence/`

In [ ]:
!pip -q install ultralytics==8.* roboflow

import sys, platform, time
from pathlib import Path
import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path.cwd()
RESULTS_DIR = ROOT / "results"
EVIDENCE_DIR = RESULTS_DIR / "evidence"
VAL_EVID_DIR = EVIDENCE_DIR / "val_preds"
NEW_EVID_DIR = EVIDENCE_DIR / "new_preds"
for p in [RESULTS_DIR, EVIDENCE_DIR, VAL_EVID_DIR, NEW_EVID_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Working directory:", ROOT)
print("Evidence folder:", EVIDENCE_DIR)

In [ ]:
from roboflow import Roboflow
from getpass import getpass

WORKSPACE = "zigurat-assignment"
PROJECT = "ppe-hardhat-detection"
VERSION = 2  # your Roboflow dataset version

RF_API_KEY = getpass("Enter your Roboflow API key (won't be saved): ")

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / "data.yaml"

print("Dataset downloaded to:", DATASET_DIR)
print("data.yaml exists:", DATA_YAML.exists())

In [ ]:
import urllib.request

WEIGHTS_URL = "PASTE_PUBLIC_DIRECT_LINK_TO_best.pt"
WEIGHTS_PATH = ROOT / "best.pt"

urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
print("Weights downloaded:", WEIGHTS_PATH.exists(), "->", WEIGHTS_PATH)

In [ ]:
from pathlib import Path
WEIGHTS_PATH = Path("best.pt")
print("Weights exist:", WEIGHTS_PATH.exists(), "->", WEIGHTS_PATH)

In [ ]:
VAL_IMG_DIR = DATASET_DIR / "valid" / "images"
val_imgs = sorted(list(VAL_IMG_DIR.glob("*")))
print("Validation images found:", len(val_imgs))

sample_val = val_imgs[:10]

# Save predictions to a temp run folder
model.predict(
    source=[str(p) for p in sample_val],
    save=True,
    imgsz=640,
    conf=0.25,
    project=str(RESULTS_DIR),
    name="val_preds_run",
    exist_ok=True
)

print("Raw val prediction outputs:", RESULTS_DIR / "val_preds_run")

In [ ]:
NEW_IMG_DIR = ROOT / "new_images"
NEW_IMG_DIR.mkdir(exist_ok=True)

new_imgs = sorted(list(NEW_IMG_DIR.glob("*")))
print("New images found:", len(new_imgs))

sample_new = new_imgs[:5]

model.predict(
    source=[str(p) for p in sample_new],
    save=True,
    imgsz=640,
    conf=0.25,
    project=str(RESULTS_DIR),
    name="new_preds_run",
    exist_ok=True
)

print("Raw new prediction outputs:", RESULTS_DIR / "new_preds_run")

In [ ]:
import shutil

VAL_RUN_DIR = RESULTS_DIR / "val_preds_run"
NEW_RUN_DIR = RESULTS_DIR / "new_preds_run"

def copy_images(src_dir: Path, dst_dir: Path, limit: int):
    imgs = sorted([p for p in src_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    for p in imgs[:limit]:
        shutil.copy(p, dst_dir / p.name)
    return [p.name for p in imgs[:limit]]

val_copied = copy_images(VAL_RUN_DIR, VAL_EVID_DIR, 10)
new_copied = copy_images(NEW_RUN_DIR, NEW_EVID_DIR, 5)

print("Copied val evidence:", val_copied)
print("Copied new evidence:", new_copied)
print("Evidence saved under:", EVIDENCE_DIR)

In [ ]:
from ultralytics import __version__ as ulty_ver

print("Reproducibility Proof (Inference Notebook)")
print("- Date/Time (UTC):", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()))
print("- Ultralytics:", ulty_ver)
print("- CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("- GPU:", torch.cuda.get_device_name(0))
print("- Dataset:", f"{WORKSPACE}/{PROJECT} v{VERSION}")
print("- Weights:", str(WEIGHTS_PATH))
print("- Evidence folder:", str(EVIDENCE_DIR))